# Training utilities: Emotion CNN với data augmentation + EarlyStopping (PyTorch)

Bản PyTorch của `Yêu cầu/examples/4. Training Utilities/3_Emotion_Reg_CNN.ipynb`. Dữ liệu và mô hình giống hệt `2. CNN + MLP/3_Emotion_Reg_CNN`; khác ở mục (3):

- Data augmentation (`ImageDataGenerator`): xoay ±45°, dịch ngang/dọc 20%, lật ngang.
- EarlyStopping: dừng khi `val_loss` không giảm 5 epoch liền, tối đa 100 epoch.

| | Keras gốc (Keras 2.0.9, GTX 1080) | PyTorch |
|---|---|---|
| Số tham số | 19.028.287 | kiểm bằng `assert` ở mục (2) |
| Dừng ở epoch | 57 (tốt nhất epoch 52) | xem mục (3) |
| Test accuracy | 0.6163 (bản thư mục 2: 0.5954) | xem mục (4) |
| Test loss | 1.0158 | xem mục (4) |
| Thời gian | 22,7 phút | xem mục (3) |

# Import libraries

Keras gốc:
```python
import keras
from keras.models import Model
from keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from keras.optimizers import Adam
from keras.preprocessing.image import ImageDataGenerator
from keras.callbacks import EarlyStopping
```

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))   # để import pytorch/common.py
from common import *

import torch.nn.functional as F
from torch.utils.data import Dataset
from scipy import ndimage
import matplotlib.pyplot as plt

# True: chỉ chạy thử (ít dữ liệu, vài epoch) để kiểm code trên CPU; bản nộp phải là False (chạy GPU)
CHAY_THU = False

set_seed(0)
device = get_device()

In [ ]:
num_classes = 7   # angry, disgust, fear, happy, neutral, sad, and surprise
TEN = "tu_3_Emotion_Reg_CNN"   # tiền tố tu_ để không trùng bản thư mục 2

# (1) Data

Giống `2. CNN + MLP/3_Emotion_Reg_CNN`: Training → train, PublicTest → "testing" (dùng làm validation như gốc), **không chia 255**.

In [ ]:
x_training, y_training = load_fer2013("Training")     # (28709, 48, 48, 1), one-hot
x_testing, y_testing = load_fer2013("PublicTest")     # (3589, 48, 48, 1)

if CHAY_THU:
    x_training, y_training = x_training[:1000], y_training[:1000]
    x_testing, y_testing = x_testing[:300], y_testing[:300]

x_test, y_test = to_nchw(x_testing), onehot_to_index(y_testing)
print(x_training.shape, tuple(x_test.shape))

# (2) Declare model

Giống hệt `2. CNN + MLP/3_Emotion_Reg_CNN` (Conv64 same, Conv64 valid, Pool, Dropout 0.25; Conv128 same, Conv128 valid, Conv218 same, Conv218 valid, Pool, Dropout 0.25; FC 1024, Dropout 0.5; FC 7).

In [ ]:
class EmoCNN(nn.Module):
    """emo_cnn: tên các lớp trùng `name=` Keras."""

    def __init__(self, num_classes=7):
        super().__init__()
        self.conv1_1 = nn.Conv2d(1, 64, 3, padding=1)      # same
        self.conv1_2 = nn.Conv2d(64, 64, 3, padding=0)     # valid
        self.pool1 = nn.MaxPool2d(2)
        self.conv_dropout1 = nn.Dropout(0.25)
        self.conv2_1 = nn.Conv2d(64, 128, 3, padding=1)    # same
        self.conv2_2 = nn.Conv2d(128, 128, 3, padding=0)   # valid
        self.conv2_3 = nn.Conv2d(128, 218, 3, padding=1)   # same
        self.conv2_4 = nn.Conv2d(218, 218, 3, padding=0)   # valid
        self.pool2 = nn.MaxPool2d(2)
        self.conv_dropout2 = nn.Dropout(0.25)
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(218 * 9 * 9, 1024)
        self.fc_dropout1 = nn.Dropout(0.5)
        self.fc3_7ways_softmax = nn.Linear(1024, num_classes)   # trả logits

    def forward(self, x):                                  # (N, 1, 48, 48), giá trị 0–255
        x = F.relu(self.conv1_1(x))                        # (N, 64, 48, 48)
        x = F.relu(self.conv1_2(x))                        # (N, 64, 46, 46)
        x = self.conv_dropout1(self.pool1(x))              # (N, 64, 23, 23)
        x = F.relu(self.conv2_1(x))                        # (N, 128, 23, 23)
        x = F.relu(self.conv2_2(x))                        # (N, 128, 21, 21)
        x = F.relu(self.conv2_3(x))                        # (N, 218, 21, 21)
        x = F.relu(self.conv2_4(x))                        # (N, 218, 19, 19)
        x = self.conv_dropout2(self.pool2(x))              # (N, 218, 9, 9)
        x = self.flatten(x)                                # (N, 17658)
        x = self.fc_dropout1(F.relu(self.fc1(x)))          # (N, 1024)
        return self.fc3_7ways_softmax(x)                   # (N, 7)


model = init_keras_like(EmoCNN(num_classes)).to(device)
summary(model, input_size=(1, 1, 48, 48))
n_params = count_params(model)
print("Số tham số PyTorch:", n_params)
assert n_params == 19_028_287

# (3) Train defined model

## Data augmentation

Keras gốc:
```python
datagen = ImageDataGenerator(
    rotation_range=45,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True)
```

Bản PyTorch viết lại đúng phép biến đổi của `ImageDataGenerator.random_transform` trong Keras 2.0.9 thay vì dùng `torchvision.transforms`:
- Góc xoay lấy đều trong [−45°, 45°]; độ dịch lấy đều trong [−0,2; 0,2] nhân với chiều cao/chiều rộng ảnh; ghép ma trận xoay rồi dịch, đặt tâm biến đổi ở giữa ảnh.
- Áp bằng `scipy.ndimage.affine_transform` với `order=0` (lấy điểm gần nhất) và `mode='nearest'` (vùng trống lấy giá trị điểm biên), như Keras.
- Lật ngang với xác suất 0,5.

`torchvision.transforms.RandomAffine` lấp vùng trống bằng 0 (màu đen) và nội suy khác, nên không dùng để giữ so sánh công bằng.

Mỗi epoch đi đúng một lượt dữ liệu train: gốc đặt `steps_per_epoch = len(x_training) / batch_size` = 224,3 → 225 bước, trong đó bước cuối có 37 ảnh, giống `DataLoader` với `batch_size=128`.

In [ ]:
def keras_random_transform(img, rotation_range=45, width_shift_range=0.2, height_shift_range=0.2,
                           horizontal_flip=True):
    """Một ảnh xám (H, W): xoay + dịch + lật ngang ngẫu nhiên như ImageDataGenerator Keras 2.0.9."""
    h, w = img.shape
    theta = np.deg2rad(np.random.uniform(-rotation_range, rotation_range))
    tx = np.random.uniform(-height_shift_range, height_shift_range) * h   # dịch theo hàng
    ty = np.random.uniform(-width_shift_range, width_shift_range) * w     # dịch theo cột
    rotation = np.array([[np.cos(theta), -np.sin(theta), 0],
                         [np.sin(theta), np.cos(theta), 0],
                         [0, 0, 1]])
    shift = np.array([[1, 0, tx], [0, 1, ty], [0, 0, 1]])
    m = rotation @ shift
    o_x, o_y = h / 2 + 0.5, w / 2 + 0.5                                   # biến đổi quanh tâm ảnh
    m = np.array([[1, 0, o_x], [0, 1, o_y], [0, 0, 1]]) @ m @ np.array([[1, 0, -o_x], [0, 1, -o_y], [0, 0, 1]])
    out = ndimage.affine_transform(img, m[:2, :2], m[:2, 2], order=0, mode='nearest')
    if horizontal_flip and np.random.random() < 0.5:
        out = out[:, ::-1]
    return np.ascontiguousarray(out)


class AugmentedFER(Dataset):
    """Thay datagen.flow(x_training, y_training): mỗi lần lấy ảnh thì biến đổi ngẫu nhiên một lần."""

    def __init__(self, x, y):
        self.x = x[..., 0]                                 # (N, 48, 48), 0–255
        self.y = onehot_to_index(y)

    def __len__(self):
        return len(self.x)

    def __getitem__(self, i):
        img = keras_random_transform(self.x[i])
        return torch.from_numpy(img).unsqueeze(0), self.y[i]   # (1, 48, 48), nhãn


# Xem thử: ảnh gốc và 5 ảnh sau augmentation
plt.figure(figsize=(12, 2.4))
plt.subplot(1, 6, 1); plt.imshow(x_training[0, :, :, 0], cmap='gray'); plt.axis('off'); plt.title('gốc')
for k in range(5):
    plt.subplot(1, 6, k + 2); plt.imshow(keras_random_transform(x_training[0, :, :, 0]), cmap='gray'); plt.axis('off')
plt.show()

## Huấn luyện

Keras gốc:
```python
loss = 'categorical_crossentropy'
lr = 0.0001
model.compile(loss=loss, optimizer=Adam(lr=lr), metrics=['accuracy'])
early_stopping_after_epochs = 5
early_stopping = EarlyStopping(monitor='val_loss', patience=early_stopping_after_epochs)
batch_size = 128
epochs = 100
history = model.fit_generator(datagen.flow(x_training, y_training, batch_size=batch_size),
                              steps_per_epoch=len(x_training) / batch_size,
                              validation_data=(x_testing, y_testing),
                              epochs=epochs,
                              callbacks=[early_stopping])
```
`fit_generator` không có trong PyTorch; `fit()` của `common.py` nhận `DataLoader` bọc `AugmentedFER`. Validation không augmentation, như gốc. EarlyStopping không khôi phục trọng số tốt nhất, nên mục (4) đánh giá mô hình ở epoch cuối, giống gốc.

In [ ]:
loss_fn = nn.CrossEntropyLoss()
lr = 0.0001
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

early_stopping_after_epochs = 1 if CHAY_THU else 5
early_stopping = EarlyStopping(monitor='val_loss', patience=early_stopping_after_epochs)

batch_size = 128
epochs = 3 if CHAY_THU else 100

train_loader = DataLoader(AugmentedFER(x_training, y_training), batch_size=batch_size, shuffle=True, num_workers=0)
test_loader = make_loader(x_test, y_test, batch_size=batch_size, shuffle=False)

starting_time = time.time()
history = fit(model, train_loader, test_loader, loss_fn, optimizer, epochs, metric=accuracy,
              callbacks=[early_stopping])
train_time = time.time() - starting_time
print('> training time is %.4f minutes' % (train_time / 60))
print('> dừng sau epoch %d, tốt nhất epoch %s' % (len(history["loss"]), early_stopping.best_epoch))

### Training time
- Gốc: GTX 1080 khoảng 22,7 phút cho 57 epoch; accuracy tăng ~2 điểm so với bản không augmentation mà không bị overfitting.
- Bản PyTorch: xem hai dòng in ở trên.

In [ ]:
fig = plot_history(history, None if CHAY_THU else TEN, metric_label="accuracy",
                   title="Emotion CNN + augmentation + EarlyStopping (PyTorch)")
plt.show()

# (4) Evaluate trained model

Kết quả gốc (mô hình ở epoch cuối 57): `Test loss: 1.0158`, `Test accuracy: 0.6163`.

In [ ]:
score = evaluate(model, test_loader, loss_fn, accuracy)
print('Test loss:', score[0])
print('Test accuracy:', score[1])

In [ ]:
if not CHAY_THU:
    save_history(history, TEN)
    save_metrics(TEN, test_loss=score[0], test_metric=score[1], n_params=n_params,
                 epochs_ran=len(history["loss"]), best_epoch=early_stopping.best_epoch,
                 train_time_s=train_time, device=device_name(device), seed=0)
    ghi_so_sanh(TEN, "4. Training Utilities", "test_acc", keras=0.6163, pytorch=score[1],
                keras_params=19028287, pytorch_params=n_params, epochs_keras=57,
                epochs_pytorch=len(history["loss"]), optimizer="Adam", lr=lr, batch=batch_size,
                device=device_name(device), train_time_s=train_time, seed=0,
                ghi_chu="Augmentation viết lại như ImageDataGenerator Keras 2.0.9 + EarlyStopping(5); "
                        "Keras dừng epoch 57; PyTorch dừng epoch %d" % len(history["loss"]))

In [ ]:
def get_emotion(ohv):
    """Như gốc: nhận vector (one-hot hoặc xác suất/logits) và trả tên cảm xúc theo argmax."""
    indx = int(np.argmax(np.asarray(ohv)))
    return ['angry', 'disgust', 'fear', 'happy', 'sad', 'surprise', 'neutral'][indx]

In [ ]:
img_indx = np.uint32(np.random.rand() * (x_test.shape[0] - 1))
sample = x_test[img_indx]                          # (1, 48, 48)

pred_cls = predict(model, sample.unsqueeze(0))

plt.imshow(sample[0], cmap='gray')
plt.show()
print('> testing image index: %d\n> true emotion: %s\n> predicted emotion: %s'
      % (img_indx, get_emotion(y_testing[img_indx, :]), get_emotion(pred_cls)))